# Data Quality & Reconciliation

Accounts for every record from raw export to what lands on the map, so you can
answer *"why isn't property X on the map?"*. The funnel:

```
raw merged rows
  - unparseable Sale Date        (dropped)
  - Sale Price < $1,000          (non-arms-length: $0-$10 transfers, dropped)
  = clean (used for trend analysis)
  - address not geocoded         (no lat/lon, can't map)
  = mappable (all years/classes)
  - outside the active FILTERS   (year / class / beds)
  = shown on the map
```

Most "missing from map" cases are the **last** step (a filter), not bad data —
e.g. *77 Sycamore Ave* is geocoded fine but its only real sale was 2013, so the
default `year_min=2020` map hides it.

In [1]:
import re
import sys
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))
from config import get_city

# --- choose the city here: "livingston" or "millburn" ---
CITY = get_city("livingston")

df = pd.read_csv(CITY.data, dtype=str)
cache = pd.read_csv(CITY.geocode_cache, dtype={'address': str})

df['sale_date'] = pd.to_datetime(df['Sale Date'], format='%m/%d/%Y', errors='coerce')
df['price'] = pd.to_numeric(df['Sale Price'], errors='coerce')
df['year'] = df['sale_date'].dt.year
df['addr'] = df['Property Address'].str.strip()

matched = set(cache.loc[cache['matched'], 'address'])

# per-row drop reasons (a row can fail more than one; each is reported).
# A non-numeric/missing price counts as non-arms-length too: NaN < 1000 is False,
# so without this such rows would slip through here but get dropped silently by
# the trend/map notebooks (median & plotting ignore NaN) -> count them honestly.
df['bad_date']     = df['sale_date'].isna()
df['non_arms']     = df['price'].isna() | (df['price'] < 1000)
df['not_geocoded'] = ~df['addr'].isin(matched)
df['clean']    = ~df['bad_date'] & ~df['non_arms']
df['mappable'] = df['clean'] & ~df['not_geocoded']
print(f"{CITY.label}: loaded {len(df):,} raw rows; {len(matched):,} addresses geocoded")

Livingston Township (07039): loaded 20,309 raw rows; 8,626 addresses geocoded


## 1. The funnel — record counts at each stage

In [2]:
n = len(df)
clean = df[df['clean']]
mappable = df[df['mappable']]
funnel = pd.DataFrame([
    ('raw merged rows', n),
    ('  - unparseable Sale Date', -df['bad_date'].sum()),
    ('  - price missing/non-numeric', -df['price'].isna().sum()),
    ('  - Sale Price < $1,000 (non-arms-length)', -(df['price'] < 1000).sum()),
    ('= clean (trend analysis)', len(clean)),
    ('  - address not geocoded', -(clean['not_geocoded'].sum())),
    ('= mappable (all years/classes)', len(mappable)),
], columns=['stage', 'rows']).set_index('stage')
funnel

,rows
stage,
raw merged rows,20309
- unparseable Sale Date,-4
- price missing/non-numeric,-3
"- Sale Price < $1,000 (non-arms-length)",-5476
= clean (trend analysis),14828
- address not geocoded,-480
= mappable (all years/classes),14348


## 2. What gets dropped, and why

Note the funnel reasons overlap (a $1 sale may also be ungeocodable); these
tallies count each reason independently against the raw rows.

In [3]:
drops = pd.DataFrame({
    'rows': [df['bad_date'].sum(), df['non_arms'].sum(),
             df['not_geocoded'].sum(), df.loc[df['clean'], 'not_geocoded'].sum()],
}, index=['unparseable date', 'price < $1,000',
          'not geocoded (any row)', 'not geocoded (among clean)'])
drops['% of raw'] = (100 * drops['rows'] / len(df)).round(1)
drops

,rows,% of raw
unparseable date,4,0.0
"price < $1,000",5479,27.0
not geocoded (any row),614,3.0
not geocoded (among clean),480,2.4


## 3. Addresses that fail to geocode

Mostly records with no house number (commercial / condo / street-only).

In [4]:
ungeo = df.loc[df['clean'] & df['not_geocoded'], 'addr']
no_number = ungeo[~ungeo.str.match(r'^\d')]
print(f"clean rows that don't geocode: {len(ungeo):,} ({ungeo.nunique():,} unique addresses)")
print(f"  of those, addresses with no leading house number: {no_number.nunique():,}")
ungeo.value_counts().head(20)

clean rows that don't geocode: 480 (304 unique addresses)
  of those, addresses with no leading house number: 20


addr
The Regency Club        29
52 Martin Rd             7
316 Eisenhower Pkwy      7
54 Old Road              6
42 Bear Brook La         5
76 Old Road              4
7 Bear Brook La          4
Livingston Square        4
22 Thames Drive          3
32 Ambrosia Ct           3
1 Marberne Ter           3
Old Short Hills Rd       3
Laurel Avenue            3
14 Thames Drive          3
667 So Orange Ave        3
34 Bear Brook La         3
405 Eisenhower Pkwy.     3
25 Thames Drive          3
17 Braeburn Ct           3
So Orange Ave            3
Name: count, dtype: int64

## 4. "Why isn't this address on the map?"

Pass an address (substring, case-insensitive) and the **active map filters** to
see, per sale, exactly which stage includes or excludes it.

In [5]:
def diagnose(query, property_class='Residential', year_min=2020, year_max=None, beds=None):
    """Explain, per sale of `query`, whether it reaches the map under these filters."""
    m = df[df['addr'].str.contains(query, case=False, na=False)].copy()
    if m.empty:
        return f"No rows match '{query}' in the raw data at all."

    def reason(r):
        if r['bad_date']:  return 'DROPPED: unparseable sale date'
        if r['non_arms']:  return 'DROPPED: price < $1,000 (non-arms-length)'
        if r['not_geocoded']: return 'DROPPED: address not geocoded'
        if property_class is not None and r['Property Class'] != property_class:
            return f"hidden by filter: class != {property_class}"
        if year_min is not None and r['year'] < year_min:
            return f'hidden by filter: year {int(r["year"])} < {year_min}'
        if year_max is not None and r['year'] > year_max:
            return f'hidden by filter: year {int(r["year"])} > {year_max}'
        if beds is not None and pd.to_numeric(r['Beds'], errors='coerce') != beds:
            return f'hidden by filter: beds != {beds}'
        return 'ON MAP'

    m['verdict'] = m.apply(reason, axis=1)
    return m.sort_values('sale_date', ascending=False)[
        ['addr', 'Sale Date', 'Sale Price', 'Property Class', 'year', 'verdict']]

diagnose('77 Sycamore')

,addr,Sale Date,Sale Price,Property Class,year,verdict
16572,77 Sycamore Avenue,01/10/2013,975000,Residential,2013.0,hidden by filter: year 2013 < 2020
16672,77 Sycamore Avenue,11/28/2012,1,Residential,2012.0,"DROPPED: price < $1,000 (non-arms-length)"


Same address, but widen the year filter — now it's on the map:

In [6]:
diagnose('77 Sycamore', year_min=None)

,addr,Sale Date,Sale Price,Property Class,year,verdict
16572,77 Sycamore Avenue,01/10/2013,975000,Residential,2013.0,ON MAP
16672,77 Sycamore Avenue,11/28/2012,1,Residential,2012.0,"DROPPED: price < $1,000 (non-arms-length)"
